# 01 Colab Exploration

최신 전처리 산출물 경로를 기준으로 데이터를 점검하는 노트북입니다. 이 단계에서는 학습보다 입력 데이터 구조, 샘플 수, 시나리오 분포, YOLO 라벨 존재 여부를 확인합니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import sys
from pathlib import Path

PROJECT_DIR = Path('/content/drive/MyDrive/sessac_project')
assert PROJECT_DIR.exists(), f'Project directory not found: {PROJECT_DIR}'

if str(PROJECT_DIR / '3_models') not in sys.path:
    sys.path.append(str(PROJECT_DIR / '3_models'))

from colab_paths import resolve_project_paths
from behavior_modeling import discover_behavior_samples, build_group_folds
from yolo_workflow import discover_yolo_bases

paths = resolve_project_paths(PROJECT_DIR)
paths

In [ ]:
records = discover_behavior_samples(
    labels_root=paths['behavior_labels'],
    landmarks_root=paths['behavior_landmarks'],
    legacy_labels_root=paths['legacy_labels'],
    legacy_landmarks_root=paths['legacy_landmarks'],
)

print('behavior samples:', len(records))
print('example sample:', records[0] if records else 'none')

In [ ]:
from collections import Counter
import pandas as pd

scenario_counts = Counter(record.scenario for record in records)
set_counts = Counter(record.set_id for record in records)

display(pd.DataFrame({'scenario': list(scenario_counts.keys()), 'count': list(scenario_counts.values())}).sort_values('scenario'))
display(pd.DataFrame({'set_id': list(set_counts.keys()), 'count': list(set_counts.values())}).sort_values('set_id').head(20))

In [ ]:
folds = build_group_folds(records, num_folds=4)
pd.DataFrame([
    {
        'fold': fold['fold_index'],
        'train_samples': len(fold['train_keys']),
        'val_samples': len(fold['val_keys']),
        'val_sets': len(fold['val_set_ids']),
    }
    for fold in folds
])

In [ ]:
open_close_bases = discover_yolo_bases(paths['yolo_images'], paths['yolo_labels_open_close'])
full_empty_bases = discover_yolo_bases(paths['yolo_images'], paths['yolo_labels_full_empty'])

print('YOLO open/close samples:', len(open_close_bases))
print('YOLO full/empty samples:', len(full_empty_bases))
print('shared sample count:', len(set(open_close_bases) & set(full_empty_bases)))

## 다음 단계

- 행동 인식 모델 후보 비교는 `02_model_comparison.ipynb`
- 최종 모델 저장까지 포함한 학습은 `03_final_model_training.ipynb`